In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

In [ ]:
# шдях де лежать файли тесту
test_resl = "ab_test_data.csv"

In [ ]:
# прочитаємо файл з результатами опросу
df_ab_test = pd.read_csv(test_resl, low_memory=False)

In [ ]:
df_ab_test.head()

In [ ]:
df_ab_test.info()

In [ ]:
df_ab_test["timestamp"] = pd.to_datetime(df_ab_test["timestamp"])

In [ ]:
df_ab_test.head()

In [ ]:
type(df_ab_test["timestamp"])

In [ ]:
df_ab_test.info()

In [ ]:
df_ab_test["timestamp"].dtype.name

In [ ]:
is_null_date = int(df_ab_test["timestamp"].isna().sum())
is_null_date

In [ ]:
df_ab_test.groupby("test_group")["conversion"].describe()

#### Що потрібно зробити:
1) Зчитай дані з файлу та виведи інформацію про результати A/B тесту:
2) Кількість користувачів в групах A та B відповідно;
3) Кількість конверсій в групах A та B відповідно;
4) Рівень конверсії в групах A та B відповідно;
5) Дату початку та дату кінця тесту, а також тривалість тесту в днях.
6) Обери статистичний критерій для тестування гіпотез на свій розсуд та обрахуй значення статистики та p-value. Перевір, чи можемо ми відхили нульову гіпотезу. Не забудь уточнити, який критерій використовуєш
7) Побудуй візуалізацію для порівняння середніх значень у групах з 95% довірчими інтервалами
8) Підготуй висновки про результати A/B тесту

In [ ]:
users_number = df_ab_test["user_id"].size
a_group_number = df_ab_test[df_ab_test["test_group"] == "a"]["user_id"].size
b_group_number = df_ab_test[df_ab_test["test_group"] == "b"]["user_id"].size
a_convers_number = df_ab_test[df_ab_test["test_group"] == "a"]["conversion"].sum()
b_convers_number = df_ab_test[df_ab_test["test_group"] == "b"]["conversion"].sum()
a_convers_rate = a_convers_number / a_group_number * 100
b_convers_rate = b_convers_number / b_group_number * 100
uplift_rate = (b_convers_rate - a_convers_rate) / a_convers_rate * 100
rpu_a = (a_convers_number * 4.99) / a_group_number
rpu_b = (b_convers_number * 4.99) / b_group_number
start_date = df_ab_test["timestamp"].dt.date.min()
end_date = df_ab_test["timestamp"].dt.date.max()
test_duration = end_date - start_date

In [ ]:
print(f"Кількість користувачів, які брали участь у тестуванні: {users_number}")
print(f"Кількість користувачів в групі А: {a_group_number}")
print(f"Кількість конверсій в групі А: {a_convers_number}")
print(f"Рівень конверсії в групі А: {round(a_convers_rate,2)} %")
print(f"Кількість користувачів в групі В: {b_group_number}")
print(f"Кількість конверсій в групі В: {b_convers_number}")
print(f"Рівень конверсії в групі В: {round(b_convers_rate,2)} %")
print(f"Дата початку тесту: {start_date}")
print(f"Дата закінчення тесту: {end_date}")
print(f"Трвалість тесту: {test_duration.days} днів")

In [ ]:
df = pd.DataFrame(
    {
        "group": ["A", "B"],
        "Count visitors": [a_group_number, b_group_number],
        "Conversion by group": [a_convers_number, b_convers_number],
        "Conversion ratio, %": [round(a_convers_rate, 2), round(b_convers_rate, 2)],
        "Uplift, %": ["", round(uplift_rate, 2)],
        "Revenue per user, $": [round(rpu_a, 2), round(rpu_b, 2)],
    }
)
df

### Перший варіант розрахунку - Обираємо статистичний критерій Стюдента для тестування гіпотез
#### Нульова гіпотеза: конверсії двох незалежних вибірок не відрізняються.



In [ ]:
alpha = 0.05

statistic, pvalue = stats.ttest_ind(
    df_ab_test[df_ab_test["test_group"] == "b"]["conversion"],
    df_ab_test[df_ab_test["test_group"] == "a"]["conversion"],
    alternative="greater",
)

print(f"t-statistic: {round(statistic, 2)}, p-value: {round(pvalue, 2)}")

if pvalue < alpha:
    print("Різниця є статистично значуща. Нульова гіпотеза може бути відкинута.")
else:
    print("Різниця не є статистично значущою. Нульова гіпотеза не може бути відкинута.")

## Другий варіант - two-proportion z-test
Чому?
* варіанти конверсії - бінарні, тобто, або оформив підписку, або ні (0/1)
* в такому випадку середня конверсія набуває ще іншого значення - як доля між тими що оформили і тими що не оформили
* і саме в такому випадку коли значення бінарні більш спеціалізованним підходом буде саме two-proportion z-test
### Але при великіх виборках результати тестів будуть майже однаковими.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# кількість покупок в кожній групі
success_b = df_ab_test[df_ab_test["test_group"] == "b"]["conversion"].sum()
success_a = df_ab_test[df_ab_test["test_group"] == "a"]["conversion"].sum()

# кількість користувачів в кожній групі
n_b = df_ab_test[df_ab_test["test_group"] == "b"]["conversion"].count()
n_a = df_ab_test[df_ab_test["test_group"] == "a"]["conversion"].count()

# H1: conversion_B > conversion_A
statistic, pvalue = proportions_ztest(
    count=[success_b, success_a],
    nobs=[n_b, n_a],
    alternative="larger"
)

print("z-statistic:", round(statistic,2))
print("p-value:", round(pvalue, 2))

if pvalue < alpha:
    print("Різниця є статистично значуща. Нульова гіпотеза може бути відкинута.")
else:
    print("Різниця не є статистично значущою. Нульова гіпотеза не може бути відкинута.")

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(
    data=df_ab_test,
    x="test_group",
    y="conversion",
    errorbar=("ci", 95),
    hue="test_group",
    palette=["#55BB00", "#AA00FF"],
    legend=False,
)
plt.title("Результати A/B-тесту")
plt.xlabel("Група")
plt.ylabel("Середні")

plt.show()

### Бонусне завдання

Побудуй графік, що відображатиме зміну конверсії в часі. І не забувай врахувати, що на цей раз в нас дійсно є дані про час здійснення івенту.

In [ ]:
# додамо новий стовбчик з дтою і відсортуємо таблицю за зростанням дати
df_ab_test["date"] = df_ab_test["timestamp"].dt.date
df_ab_test = df_ab_test.sort_values("date")
df_ab_test.head()

In [ ]:
# розрахуємо щоденну конверсію, так як конверсія або 0 або 1, то рахуємо середню - тобто долю одиниць в загальній кількості
daily_cr = df_ab_test.groupby(["test_group", "date"])["conversion"].mean()
daily_cr

In [ ]:
pivot_daily_cr = pd.pivot_table(
    df_ab_test,
    index="test_group",
    columns="date",
    values="conversion",
    aggfunc="mean",
    fill_value=0,
    dropna=False,
)

pivot_daily_cr

In [ ]:
# побудуємо простий графік як змінялася конверсія щодня під час проведення тесту
plt.figure(figsize=(8, 6))
plt.plot(pivot_daily_cr.columns, pivot_daily_cr.loc["a"], label="A", color="#55BB00")
plt.plot(pivot_daily_cr.columns, pivot_daily_cr.loc["b"], label="B", color="#AA00FF")

plt.xlabel("Дата")
plt.ylabel("Рівень конверсії")
plt.title("Щоденна зміна конверсії за час проведення А/В -тесту")
# plt.xticks(rotation=45)
plt.legend()

plt.show()

In [ ]:
# Рахуємо кумулятивне середнє - це і є зміна конверсії з плином часу
df_ab_test = df_ab_test.sort_values("timestamp")
cumulative_metric_a = (
    df_ab_test[df_ab_test["test_group"] == "a"]["conversion"]
    .expanding()
    .mean()
    .reset_index(drop=True)
)
cumulative_metric_b = (
    df_ab_test[df_ab_test["test_group"] == "b"]["conversion"]
    .expanding()
    .mean()
    .reset_index(drop=True)
)

plt.figure(figsize=(10, 6))
plt.plot(cumulative_metric_a, label="A", color="#55BB00")
plt.plot(cumulative_metric_b, label="B", color="#AA00FF")

plt.title("Порівняння сукупного коефіцієнта конверсії")
plt.xlabel("Кумулятивна кількість користувачів (с плином часу)")
plt.ylabel("Кумулятивний рівень конверсії")

plt.legend()
plt.show()